In [2]:
import numpy as np
import pandas as pd
from ScFormer.utils import *
from ScFormer.model import *
from warnings import filterwarnings
import random
import os
import torch
import torch.cuda as cuda
from scipy import sparse

In [3]:
filterwarnings("ignore")
seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

In [4]:
gene_cell = sparse.load_npz('data/example/RNA.npz')
gene_names = pd.DataFrame(np.load('data/example/gene_name.npy', allow_pickle=True))
true_label = np.load('data/example/label500.npy', allow_pickle=True)

gene_cell.obs_names = gene_names[0]

RNA_matrix = gene_cell

cell_num = RNA_matrix.shape[1]
gene_num = RNA_matrix.shape[0]

In [5]:
initial_pre = initial_clustering(RNA_matrix)

	When the number of cells is less than or equal to 500, it is recommended to set the resolution value to 0.2.
	When the number of cells is within the range of 500 to 5000, the resolution value should be set to 0.5.
	When the number of cells is greater than 5000, the resolution value should be set to 0.8.
         Falling back to preprocessing with `sc.pp.pca` and default params.


In [6]:
cluster_ini_num = len(set(initial_pre))
ini_p1 = [int(i) for i in initial_pre]
# partite the data into batches
indices, Node_Ids, dic = batch_select_whole(RNA_matrix)
n_batch = len(indices)

Partitioning the data into batches. Please wait...


Processing Batches: 100%|██████████| 17/17 [00:00<00:00, 23.68it/s]


In [7]:
device = torch.device("cuda" if cuda.is_available() else "cpu")
node_model = NodeDimensionReduction(RNA_matrix, indices, ini_p1, n_hid=104, n_heads=8,
                                    n_layers=3, labsm=0.1, lr=0.0005, wd=0.1, device=device, num_types=2,
                                    num_relations=2, epochs=100)
gnn, cell_emb, gene_emb, h = node_model.train_model(n_batch=n_batch)

The training process for the NodeDimensionReduction model has started. Please wait.


100%|██████████| 100/100 [01:16<00:00,  1.31it/s]

The training for the NodeDimensionReduction model has been completed.


In [8]:
ScFormer_model = ScFormer(gnn=gnn, h=h, labsm=0.1, n_hid=104, n_batch=n_batch, device=device, lr=0.0005, wd=0.1,
                      num_epochs=50)
ScFormer_gnn, _, _, _ = ScFormer_model.train_model(indices=indices, RNA_matrix=RNA_matrix, ini_p1=ini_p1)

The training process for the ScFormer model has started. Please wait.
Forward pass started.


Epochs: 100%|██████████| 50/50 [00:42<00:00,  1.18it/s] 

The training for the ScFormer model has been completed.
The training for the ScFormer model has been completed.


In [10]:
ScFormer_result = ScFormer_pred(RNA_matrix, gnn=ScFormer_gnn, indices=indices,
                            nodes_id=Node_Ids, device=device)
# Save numpy arrays to files
output_file = 'data/example/output'
np.save(output_file + "/Node_Ids.npy", Node_Ids)
np.save(output_file + "/pred.npy", ScFormer_result['pred_label'])
np.save(output_file + "/cell_embedding.npy", ScFormer_result['cell_embedding'])

Prediction Batches: 100%|██████████| 17/17 [00:00<00:00, 28.70it/s]


In [11]:
pred_label = ScFormer_result['pred_label']
p_score, labels = purity_score(np.array(true_label), pred_label)
e = Entropy(np.array(pred_label, dtype='int64'), np.array(labels, dtype='int64'))
print("purity:%.4f" % p_score)
print("NMI:%.4f" % normalized_mutual_info_score(true_label, labels))
print("Entropy:%.4f" % e)

purity:0.9800
NMI:1.0000
Entropy:0.0973
